# Fine-tune `VietAI/vit5-large` bằng LoRA cho bài toán tóm tắt tiếng Việt

Notebook này được dọn lại theo pipeline gọn hơn:

1. Cài thư viện và khai báo cấu hình.
2. Đọc/làm sạch `train.parquet` và `val.parquet`.
3. Load ViT5-large, gắn LoRA, tokenize dữ liệu.
4. Fine-tune bằng `Seq2SeqTrainer`.
5. Generate trên validation, tính ROUGE/BLEU, lưu kết quả.
6. Để sẵn các cell đánh giá `test.parquet`, nhưng đã comment toàn bộ vì hiện chưa có file test.
7. Nén output để tải về.

> Khi chỉ muốn kiểm tra pipeline, giữ `DEBUG_MODE=True`.  
> Khi train thật, đổi `DEBUG_MODE=False`.

## Cell 1 — Cài thư viện

**Nhiệm vụ:** Cài các thư viện cần cho toàn bộ notebook.

**Ý nghĩa:** Cell này chỉ chạy một lần ở đầu Colab. Sau khi cài xong, các cell sau mới import được `transformers`, `peft`, `datasets`, `evaluate`, tokenizer ViT5 và các metric.

In [1]:
%%capture
# NHIỆM VỤ:
# - Cài thư viện cho fine-tune ViT5-large bằng LoRA.
# - %%capture giúp ẩn log pip dài để notebook gọn hơn.

!pip install \
    transformers==4.39.3 \
    peft==0.10.0 \
    accelerate==0.27.2 \
    datasets \
    evaluate \
    rouge-score \
    sacrebleu \
    sentencepiece \
    pyarrow \
    nltk \
    bert-score

# Ý nghĩa từng package:
# transformers==4.39.3:
#   Dùng để load tokenizer, model ViT5, Seq2SeqTrainer và TrainingArguments.
# peft==0.10.0:
#   Dùng để fine-tune bằng LoRA, chỉ train một phần nhỏ tham số.
# accelerate==0.27.2:
#   Hỗ trợ Trainer chạy GPU, FP16 và phân phối thiết bị ổn định hơn.
# datasets:
#   Chuyển pandas DataFrame sang Hugging Face Dataset và map tokenizer.
# evaluate:
#   Load metric như ROUGE và SacreBLEU.
# rouge-score:
#   Backend cần thiết để evaluate tính ROUGE.
# sacrebleu:
#   Backend cần thiết để evaluate tính BLEU.
# sentencepiece:
#   Tokenizer gốc của các model T5/ViT5.
# pyarrow:
#   Đọc file .parquet.
# nltk:
#   Thư viện NLP phụ trợ, một số metric/pipeline có thể cần.
# bert-score:
#   Metric đánh giá tương đồng ngữ nghĩa, để tùy chọn vì chạy chậm hơn ROUGE/BLEU.

## Cell 2 — Import thư viện và khai báo cấu hình toàn cục

**Nhiệm vụ:** Gom toàn bộ cấu hình chính vào một nơi: đường dẫn dữ liệu, chế độ debug/full train, độ dài token, tham số train, tham số generate và đường dẫn output.

**Ý nghĩa:** Khi muốn đổi cách chạy, thường chỉ cần sửa cell này.

In [2]:
# NHIỆM VỤ:
# - Import thư viện.
# - Khai báo toàn bộ cấu hình chính của pipeline.
# - Tạo các thư mục output cần thiết.

import os                 # Làm việc với đường dẫn, biến môi trường, tạo thư mục.
import json               # Lưu metric ra file .json.
import inspect            # Kiểm tra tham số của TrainingArguments để tương thích nhiều version.
from pathlib import Path  # Xử lý đường dẫn kiểu object, tiện khi nén output.

import numpy as np        # Xử lý mảng, thay label -100 khi decode metric.
import pandas as pd       # Đọc/làm sạch dữ liệu .parquet.
import torch              # Framework deep learning chính.
import evaluate           # Load metric ROUGE/BLEU.

from datasets import Dataset  # Chuyển DataFrame sang Hugging Face Dataset.

from transformers import (
    AutoTokenizer,              # Load tokenizer tương ứng với MODEL_NAME.
    AutoModelForSeq2SeqLM,      # Load model encoder-decoder cho bài toán seq2seq.
    DataCollatorForSeq2Seq,     # Padding động cho batch seq2seq.
    Seq2SeqTrainer,             # Trainer chuyên cho bài toán sinh chuỗi.
    Seq2SeqTrainingArguments,   # Chứa toàn bộ tham số train/eval/save/log.
    set_seed,                   # Cố định seed để kết quả dễ tái lập hơn.
)

from peft import (
    LoraConfig,       # Cấu hình LoRA.
    get_peft_model,   # Gắn LoRA vào base model.
    TaskType,         # Khai báo loại task cho PEFT.
)

# Tắt Weights & Biases để Colab không hỏi đăng nhập.
# "true": vô hiệu hóa logging wandb.
os.environ["WANDB_DISABLED"] = "true"

# Tắt tokenizer parallelism để giảm warning và tránh treo tiến trình khi map dữ liệu.
# "false": không dùng song song nội bộ của tokenizer.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Cố định seed.
# 42: giá trị phổ biến giúp sampling/debug/training ổn định giữa các lần chạy.
set_seed(42)

# =========================
# 1. Cấu hình model
# =========================

# Tên model trên Hugging Face.
# VietAI/vit5-large: model T5 tiếng Việt kích thước large.
MODEL_NAME = "VietAI/vit5-large"

# Prefix thêm vào đầu article theo phong cách T5 text-to-text.
# Prefix giúp model hiểu nhiệm vụ là tóm tắt tiếng Việt.
TASK_PREFIX = "vietnamese summary: "

# =========================
# 2. Cấu hình đường dẫn Colab
# =========================

# Thư mục gốc của project trên Colab.
# Cấu trúc mong muốn:
# /content/NLP/
# ├── data/
# │   ├── train.parquet
# │   └── val.parquet
# └── output/
PROJECT_DIR = "/content/NLP"

# Thư mục chứa dữ liệu.
DATA_DIR = f"{PROJECT_DIR}/data"

# Thư mục gốc chứa toàn bộ output.
OUTPUT_BASE_DIR = f"{PROJECT_DIR}/output"

# Thư mục lưu LoRA adapter, tokenizer và checkpoint cuối.
OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/vit5-large-lora"

# File train.
TRAIN_PATH = f"{DATA_DIR}/train.parquet"

# File validation.
VALID_PATH = f"{DATA_DIR}/val.parquet"

# File CSV lưu prediction trên validation.
PREDICTION_CSV = f"{OUTPUT_BASE_DIR}/vit5_large_validation_predictions.csv"

# File JSON lưu ROUGE/BLEU validation.
METRICS_JSON = f"{OUTPUT_BASE_DIR}/vit5_large_validation_metrics.json"

# File JSON lưu BERTScore nếu bật RUN_BERTSCORE=True.
BERTSCORE_JSON = f"{OUTPUT_BASE_DIR}/vit5_large_validation_bertscore.json"

# File CSV lưu ví dụ định tính cho báo cáo.
EXAMPLES_CSV = f"{OUTPUT_BASE_DIR}/vit5_large_examples_for_report.csv"

# File zip gom output để tải về.
ZIP_PATH = f"{OUTPUT_BASE_DIR}/vit5_large_outputs.zip"

# Tạo thư mục nếu chưa tồn tại.
# exist_ok=True: nếu thư mục đã có thì không báo lỗi.
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# 3. Cấu hình cột dữ liệu
# =========================

# Cột chứa văn bản gốc cần tóm tắt.
SOURCE_COL = "article"

# Cột chứa tóm tắt chuẩn.
TARGET_COL = "summary"

# =========================
# 4. Chế độ debug/full train
# =========================

# True:
#   Chỉ lấy một phần nhỏ dữ liệu để kiểm tra pipeline nhanh.
# False:
#   Dùng toàn bộ dữ liệu để train thật.
DEBUG_MODE = False

# Số mẫu train khi DEBUG_MODE=True.
# 500 đủ để kiểm tra code chạy, chưa dùng để kết luận chất lượng model.
DEBUG_TRAIN_SIZE = 500

# Số mẫu validation khi DEBUG_MODE=True.
# 100 đủ để kiểm tra eval/generate nhanh.
DEBUG_VALID_SIZE = 100

# =========================
# 5. Cấu hình tokenize
# =========================

# Số token tối đa của article đầu vào.
# 768 giữ được nhiều nội dung hơn 512 nhưng tốn VRAM hơn.
# Nếu bị OOM trên T4, giảm xuống 512.
MAX_SOURCE_LENGTH = 768

# Số token tối đa của summary đầu ra.
# 160 phù hợp summary tiếng Việt mức vừa.
# Nếu output quá dài/copy nhiều, có thể giảm xuống 128.
MAX_TARGET_LENGTH = 160

# Số tiến trình khi tokenization.
# 2 thường ổn trên Colab; nếu lỗi RAM/multiprocessing thì đổi thành 1.
TOKENIZE_NUM_PROC = 2

# =========================
# 6. Cấu hình train
# =========================

# Số epoch train.
# 3 là baseline hợp lý; tăng epoch có thể cải thiện nhưng tốn thời gian và dễ overfit.
NUM_TRAIN_EPOCHS = 5

# Learning rate.
# 2e-4 thường phù hợp LoRA vì chỉ train ít tham số.
LEARNING_RATE = 2e-4

# Batch size train trên mỗi GPU.
# 1 an toàn cho ViT5-large trên GPU T4 16GB.
PER_DEVICE_TRAIN_BATCH_SIZE = 1

# Batch size eval trên mỗi GPU.
# 1 an toàn khi eval model large.
PER_DEVICE_EVAL_BATCH_SIZE = 1

# Số step cộng dồn gradient trước khi update.
# Effective batch size = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS.
# 1 * 8 = 8.
GRADIENT_ACCUMULATION_STEPS = 8

# Bật mixed precision FP16.
# True giúp giảm VRAM và tăng tốc trên T4.
FP16 = True

# Bật gradient checkpointing.
# True tiết kiệm VRAM bằng cách tính lại một số activation lúc backward.
GRADIENT_CHECKPOINTING = True

# Có generate summary trong lúc Trainer evaluation hay không.
# False: eval nhanh, chỉ tính eval_loss trong lúc train.
# True: eval chậm hơn nhưng có ROUGE từng epoch.
PREDICT_WITH_GENERATE_DURING_TRAINING = False

# Số beam khi generate trong evaluation của Trainer nếu bật predict_with_generate.
# 1 nhanh nhất, gần greedy decoding.
GENERATION_NUM_BEAMS_DURING_TRAINING = 1

# =========================
# 7. Cấu hình generate cuối cùng
# =========================

# Batch size khi generate prediction cuối.
# 4 thường chạy được trên T4; nếu OOM thì giảm xuống 2 hoặc 1.
FINAL_GENERATE_BATCH_SIZE = 4

# Số beam khi generate cuối.
# 4 cân bằng giữa chất lượng và tốc độ.
FINAL_NUM_BEAMS = 4

# Độ dài tối thiểu của summary sinh ra.
# 30 giúp tránh output quá ngắn.
# Nếu model hay copy dài, có thể giảm 20-25.
FINAL_MIN_LENGTH = 30

# Cấm lặp lại n-gram trong output.
# 3 giúp giảm lỗi lặp cụm từ.
FINAL_NO_REPEAT_NGRAM_SIZE = 3

# Số mẫu validation generate cuối.
# None: generate toàn bộ validation.
# Nếu DEBUG_MODE=True, cell generate sẽ tự giới hạn để chạy nhanh.
VALIDATION_MAX_GENERATE_SAMPLES = None

# Bật/tắt BERTScore.
# False vì BERTScore chậm; bật True nếu cần metric ngữ nghĩa cho báo cáo.
RUN_BERTSCORE = False

# =========================
# 8. Kiểm tra GPU và đường dẫn
# =========================

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Train path:", TRAIN_PATH)
print("Validation path:", VALID_PATH)
print("Output directory:", OUTPUT_DIR)

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Project directory: /content/NLP
Data directory: /content/NLP/data
Train path: /content/NLP/data/train.parquet
Validation path: /content/NLP/data/val.parquet
Output directory: /content/NLP/output/vit5-large-lora


## Cell 3 — Kiểm tra file dữ liệu

**Nhiệm vụ:** Kiểm tra `train.parquet` và `val.parquet` có tồn tại đúng đường dẫn không.

**Ý nghĩa:** Dừng sớm nếu sai path, tránh lỗi khó hiểu ở các cell sau.

In [3]:
# NHIỆM VỤ:
# - Kiểm tra 2 file bắt buộc trước khi đọc dữ liệu.

# Danh sách path bắt buộc.
required_paths = [TRAIN_PATH, VALID_PATH]

# Lọc ra các path chưa tồn tại.
missing_paths = [p for p in required_paths if not os.path.exists(p)]

# Nếu thiếu file, dừng notebook và báo rõ file nào bị thiếu.
if missing_paths:
    raise FileNotFoundError(
        "Thiếu file dữ liệu. Hãy kiểm tra lại cấu trúc thư mục Colab:\n"
        + "\n".join(missing_paths)
    )

print("Đã tìm thấy đầy đủ file dữ liệu:")
for p in required_paths:
    print("-", p)

Đã tìm thấy đầy đủ file dữ liệu:
- /content/NLP/data/train.parquet
- /content/NLP/data/val.parquet


## Cell 4 — Đọc và làm sạch dữ liệu

**Nhiệm vụ:** Đọc `train.parquet` và `val.parquet`, kiểm tra cột bắt buộc, xóa dòng rỗng, chuẩn hóa khoảng trắng.

**Ý nghĩa:** Đảm bảo dữ liệu đầu vào chỉ còn hai cột sạch: `article` và `summary`.

In [4]:
# NHIỆM VỤ:
# - Đọc dữ liệu parquet.
# - Kiểm tra cột article/summary.
# - Làm sạch text và loại bỏ dòng không hợp lệ.

def clean_text(text):
    """Chuẩn hóa nhẹ văn bản.

    Parameters:
    - text:
        Giá trị đầu vào, có thể là string hoặc kiểu khác.

    Returns:
    - String đã được ép kiểu, xóa khoảng trắng đầu/cuối và gộp nhiều khoảng trắng liên tiếp.
    """

    # str(text): ép mọi giá trị về string.
    # split(): tách theo mọi khoảng trắng, tự loại khoảng trắng thừa.
    # " ".join(...): ghép lại bằng đúng một dấu cách.
    return " ".join(str(text).split())


def load_and_clean_parquet(path, source_col=SOURCE_COL, target_col=TARGET_COL):
    """Đọc và làm sạch một file parquet.

    Parameters:
    - path:
        Đường dẫn file .parquet cần đọc.
    - source_col:
        Tên cột chứa article đầu vào.
    - target_col:
        Tên cột chứa summary chuẩn.

    Returns:
    - DataFrame chỉ gồm source_col và target_col đã được làm sạch.
    """

    # Đọc parquet thành pandas DataFrame.
    df = pd.read_parquet(path)

    # In thông tin ban đầu để kiểm tra nhanh.
    print(f"\nFile: {path}")
    print("Shape gốc:", df.shape)
    print("Columns:", df.columns.tolist())

    # Kiểm tra cột bắt buộc.
    required_cols = [source_col, target_col]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(
                f"Thiếu cột {col!r} trong {path}. "
                f"Các cột hiện có: {df.columns.tolist()}"
            )

    # Chỉ giữ 2 cột cần thiết.
    # dropna(): xóa dòng có NaN ở article hoặc summary.
    # copy(): tạo bản sao để tránh pandas warning.
    df = df[required_cols].dropna().copy()

    # Chuẩn hóa khoảng trắng.
    df[source_col] = df[source_col].map(clean_text)
    df[target_col] = df[target_col].map(clean_text)

    # Loại dòng rỗng sau khi clean.
    df = df[
        (df[source_col].str.len() > 0)
        & (df[target_col].str.len() > 0)
    ].reset_index(drop=True)

    print("Shape sau làm sạch:", df.shape)
    return df


# Đọc và làm sạch train/validation.
train_df = load_and_clean_parquet(TRAIN_PATH)
valid_df = load_and_clean_parquet(VALID_PATH)

# Hiển thị vài dòng đầu để kiểm tra nội dung.
display(train_df.head(3))


File: /content/NLP/data/train.parquet
Shape gốc: (10775, 2)
Columns: ['article', 'summary']
Shape sau làm sạch: (10775, 2)

File: /content/NLP/data/val.parquet
Shape gốc: (1349, 2)
Columns: ['article', 'summary']
Shape sau làm sạch: (1349, 2)


,article,summary
0,Gần 20 sự kiện được tổ chức trên toàn thành ph...,Hà Nội tổ chức gần 20 sự kiện từ 19/4 đến 10/5...
1,"Được thành lập năm 1897 tại Đức, Kempinski Hot...",Kempinski Hotels là một thương hiệu nổi tiếng ...
2,"Ngoài di chuyển đến Tuần Châu bằng đường bộ, m...",Bài viết giới thiệu các hoạt động vui chơi giả...


## Cell 5 — Chọn debug/full dataset và chuyển sang Hugging Face Dataset

**Nhiệm vụ:** Nếu `DEBUG_MODE=True`, lấy mẫu nhỏ để chạy nhanh. Nếu `DEBUG_MODE=False`, dùng toàn bộ dữ liệu.

**Ý nghĩa:** Debug trước giúp kiểm tra lỗi path, tokenizer, model và OOM trước khi train thật.

In [5]:
# NHIỆM VỤ:
# - Quyết định số mẫu dùng để train/evaluate.
# - Chuyển pandas DataFrame sang Hugging Face Dataset.

if DEBUG_MODE:
    # sample(..., random_state=42): lấy mẫu ngẫu nhiên nhưng tái lập được.
    # min(...): tránh lỗi nếu dataset nhỏ hơn số mẫu debug.
    train_df = train_df.sample(
        n=min(DEBUG_TRAIN_SIZE, len(train_df)),
        random_state=42,
    ).reset_index(drop=True)

    valid_df = valid_df.sample(
        n=min(DEBUG_VALID_SIZE, len(valid_df)),
        random_state=42,
    ).reset_index(drop=True)

    print("DEBUG_MODE=True: đang dùng tập nhỏ để kiểm tra pipeline.")
else:
    # Full train: dùng toàn bộ dữ liệu sau cleaning.
    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)

    print("DEBUG_MODE=False: đang dùng toàn bộ dataset để train thật.")

print("Train used:", train_df.shape)
print("Valid used:", valid_df.shape)

# Chuyển DataFrame sang Hugging Face Dataset.
# preserve_index=False: không đưa index pandas thành cột thừa.
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
valid_dataset = Dataset.from_pandas(valid_df, preserve_index=False)

print(train_dataset)
print(valid_dataset)

DEBUG_MODE=False: đang dùng toàn bộ dataset để train thật.
Train used: (10775, 2)
Valid used: (1349, 2)
Dataset({
    features: ['article', 'summary'],
    num_rows: 10775
})
Dataset({
    features: ['article', 'summary'],
    num_rows: 1349
})


## Cell 6 — Load tokenizer và base model

**Nhiệm vụ:** Tải tokenizer và model `VietAI/vit5-large`.

**Ý nghĩa:** Tokenizer chuyển tiếng Việt thành token ID; model seq2seq học ánh xạ từ `article` sang `summary`.

In [6]:
# NHIỆM VỤ:
# - Load tokenizer và base model ViT5-large.

# Load tokenizer.
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,      # Tên tokenizer/model trên Hugging Face.
    use_fast=False,  # False: dùng SentencePiece tokenizer chuẩn của ViT5, ổn định hơn.
)

# Load base model seq2seq.
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,               # Tên model trên Hugging Face.
    torch_dtype=torch.float32  # Load float32; Trainer sẽ dùng autocast FP16 khi FP16=True.
)

print("Tokenizer loaded from:", MODEL_NAME)
print("Tokenizer vocab size:", len(tokenizer))
print("Model vocab size:", model.config.vocab_size)

# Kiểm tra tokenizer và model có cùng vocab size.
# Nếu lệch, input_ids có thể vượt quá embedding size của model.
assert len(tokenizer) == model.config.vocab_size, (
    f"Tokenizer vocab size {len(tokenizer)} != model vocab size {model.config.vocab_size}"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used s

Tokenizer loaded from: VietAI/vit5-large
Tokenizer vocab size: 36100
Model vocab size: 36100


## Cell 7 — Gắn LoRA adapter

**Nhiệm vụ:** Gắn LoRA vào các module attention của ViT5-large.

**Ý nghĩa:** Chỉ train một lượng nhỏ tham số LoRA thay vì cập nhật toàn bộ model gần 794M tham số, phù hợp GPU T4.

In [7]:
# NHIỆM VỤ:
# - Cấu hình LoRA.
# - Gắn LoRA vào base model.
# - Kiểm tra tỷ lệ tham số trainable.

# Tắt cache vì cache không tương thích tốt với gradient checkpointing khi train.
# False: không lưu key/value cache của decoder trong training.
model.config.use_cache = False

# Khai báo cấu hình LoRA.
lora_config = LoraConfig(
    r=8,
    # r:
    #   Rank của ma trận LoRA.
    #   Rank càng lớn thì càng nhiều tham số trainable và có thể học tốt hơn nhưng tốn VRAM hơn.

    lora_alpha=16,
    # lora_alpha:
    #   Hệ số scale cho LoRA.
    #   Thường đặt khoảng 2*r, ở đây r=8 nên alpha=16.

    target_modules=["q", "v"],
    # target_modules:
    #   Các module được gắn LoRA.
    #   "q" = query projection, "v" = value projection trong attention.
    #   Đây là lựa chọn nhẹ, thường đủ tốt cho fine-tune seq2seq.

    lora_dropout=0.05,
    # lora_dropout:
    #   Dropout trên nhánh LoRA.
    #   Giúp giảm overfitting, đặc biệt khi dataset không quá lớn.

    bias="none",
    # bias:
    #   "none" nghĩa là không train bias.
    #   Giảm số tham số và giảm VRAM.

    task_type=TaskType.SEQ_2_SEQ_LM,
    # task_type:
    #   Khai báo bài toán sequence-to-sequence language modeling.
)

# Gắn LoRA vào model.
# Sau bước này, phần lớn tham số base model bị freeze, chỉ LoRA được train.
model = get_peft_model(model, lora_config)

# Quan trọng khi dùng LoRA + gradient checkpointing.
# Bật gradient cho input embedding để backward hoạt động đúng.
model.enable_input_require_grads()

# In tổng quan số tham số trainable.
model.print_trainable_parameters()

trainable params: 2,359,296 || all params: 793,644,032 || trainable%: 0.2972738286778927


## Cell 8 — Tokenize dữ liệu

**Nhiệm vụ:** Chuyển `article` và `summary` thành token ID để đưa vào Trainer.

**Ý nghĩa:** Input được thêm `TASK_PREFIX`; padding để động ở bước tạo batch nhằm tiết kiệm VRAM.

In [8]:
# NHIỆM VỤ:
# - Tokenize article và summary.
# - Tạo cột input_ids, attention_mask và labels.

def preprocess_function(examples):
    """Chuyển một batch text thành token ID.

    Parameters:
    - examples:
        Một batch từ Hugging Face Dataset, gồm SOURCE_COL và TARGET_COL.

    Returns:
    - model_inputs:
        Dictionary gồm input_ids, attention_mask và labels.
    """

    # Thêm prefix nhiệm vụ vào từng article.
    # T5 học theo format text-to-text, nên prefix giúp model hiểu đây là tác vụ tóm tắt.
    inputs = [TASK_PREFIX + clean_text(doc) for doc in examples[SOURCE_COL]]

    # Summary chuẩn làm target.
    targets = [clean_text(t) for t in examples[TARGET_COL]]

    # Tokenize input article.
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LENGTH,
        # max_length:
        #   Giới hạn số token article.
        #   Nếu article dài hơn, phần dư bị cắt.

        truncation=True,
        # truncation:
        #   True nghĩa là cắt input quá dài để không vượt giới hạn model/VRAM.

        padding=False,
        # padding:
        #   False ở bước map để chưa padding toàn dataset.
        #   Padding sẽ làm động theo từng batch ở DataCollatorForSeq2Seq.
    )

    # Tokenize target summary.
    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        # max_length:
        #   Giới hạn số token summary chuẩn.

        truncation=True,
        # truncation:
        #   Cắt summary nếu quá dài.

        padding=False,
        # padding:
        #   Không padding sớm để tiết kiệm RAM.
    )

    # Trainer dùng labels để tính loss.
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


# Tokenize train dataset.
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    # batched:
    #   True nghĩa là xử lý theo batch, nhanh hơn xử lý từng dòng.

    remove_columns=train_dataset.column_names,
    # remove_columns:
    #   Xóa cột text gốc sau tokenize để giảm RAM.

    num_proc=TOKENIZE_NUM_PROC,
    # num_proc:
    #   Số tiến trình song song khi tokenize.
)

# Tokenize validation dataset.
tokenized_valid = valid_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=valid_dataset.column_names,
    num_proc=TOKENIZE_NUM_PROC,
)

print(tokenized_train)
print(tokenized_valid)

Map (num_proc=2):   0%|          | 0/10775 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/1349 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 10775
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1349
})


## Cell 9 — Data collator và metric ROUGE trong Trainer

**Nhiệm vụ:** Tạo collator padding động và hàm tính ROUGE nếu bật generate trong evaluation.

**Ý nghĩa:** Collator giúp batch có cùng độ dài; metric function dùng khi `PREDICT_WITH_GENERATE_DURING_TRAINING=True`.

In [9]:
# NHIỆM VỤ:
# - Tạo DataCollatorForSeq2Seq.
# - Tạo hàm compute_metrics cho Trainer.

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    # tokenizer:
    #   Dùng để biết pad_token_id và cách padding.

    model=model,
    # model:
    #   Model seq2seq, giúp collator chuẩn bị decoder input nếu cần.

    label_pad_token_id=-100,
    # label_pad_token_id:
    #   -100 khiến loss bỏ qua vị trí padding của labels.
)

# Load metric ROUGE.
# rouge1: trùng unigram.
# rouge2: trùng bigram.
# rougeL: longest common subsequence.
rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    """Tính ROUGE khi Trainer generate prediction trong evaluation.

    Parameters:
    - eval_preds:
        Tuple gồm predictions và labels từ Trainer.

    Returns:
    - Dictionary metric ROUGE chính.
    """

    preds, labels = eval_preds

    # Một số version Trainer trả prediction dạng tuple.
    if isinstance(preds, tuple):
        preds = preds[0]

    # Decode prediction token ID thành text.
    decoded_preds = tokenizer.batch_decode(
        preds,
        skip_special_tokens=True,
        # skip_special_tokens:
        #   Bỏ token đặc biệt như <pad>, </s>.

        clean_up_tokenization_spaces=True,
        # clean_up_tokenization_spaces:
        #   Dọn khoảng trắng thừa sau decode.
    )

    # Label padding đang là -100, cần đổi về pad_token_id để tokenizer decode được.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Decode labels thành text reference.
    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    # Strip để metric ổn định hơn.
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    # Tính ROUGE.
    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=False,
        # use_stemmer:
        #   False vì tiếng Việt không dùng stemming kiểu tiếng Anh.
    )

    return {
        "rouge1": result["rouge1"],
        "rouge2": result["rouge2"],
        "rougeL": result["rougeL"],
    }

## Cell 10 — Cấu hình `Seq2SeqTrainingArguments`

**Nhiệm vụ:** Tạo object cấu hình cho Trainer: batch size, epoch, learning rate, FP16, save/eval/logging.

**Ý nghĩa:** Đây là cell quyết định cách model được huấn luyện.

In [10]:
# NHIỆM VỤ:
# - Tạo Seq2SeqTrainingArguments.
# - Tương thích cả version transformers dùng eval_strategy và evaluation_strategy.

def build_training_args():
    """Tạo TrainingArguments cho Seq2SeqTrainer.

    Returns:
    - Seq2SeqTrainingArguments:
        Object chứa toàn bộ cấu hình train/eval/save/log.
    """

    common_kwargs = dict(
        output_dir=OUTPUT_DIR,
        # output_dir:
        #   Nơi Trainer lưu checkpoint và log.

        do_train=True,
        # do_train:
        #   True cho phép gọi trainer.train().

        do_eval=True,
        # do_eval:
        #   True cho phép đánh giá trên validation.

        learning_rate=LEARNING_RATE,
        # learning_rate:
        #   Tốc độ cập nhật tham số LoRA.

        num_train_epochs=NUM_TRAIN_EPOCHS,
        # num_train_epochs:
        #   Số lần model đi qua toàn bộ train set.

        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        # per_device_train_batch_size:
        #   Batch size train trên mỗi GPU.

        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        # per_device_eval_batch_size:
        #   Batch size eval trên mỗi GPU.

        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        # gradient_accumulation_steps:
        #   Cộng dồn gradient nhiều step rồi mới update.
        #   Giúp tăng effective batch size mà không tăng VRAM.

        predict_with_generate=PREDICT_WITH_GENERATE_DURING_TRAINING,
        # predict_with_generate:
        #   True thì Trainer generate summary khi eval để tính ROUGE.
        #   False thì eval nhanh hơn, chỉ tính eval_loss.

        generation_max_length=MAX_TARGET_LENGTH,
        # generation_max_length:
        #   Độ dài tối đa khi generate trong eval của Trainer.

        generation_num_beams=GENERATION_NUM_BEAMS_DURING_TRAINING,
        # generation_num_beams:
        #   Số beam khi generate trong eval của Trainer.

        fp16=FP16,
        # fp16:
        #   Bật mixed precision để giảm VRAM/tăng tốc trên T4.

        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        # gradient_checkpointing:
        #   Tiết kiệm VRAM bằng cách tính lại activation khi backward.

        save_strategy="epoch",
        # save_strategy:
        #   Lưu checkpoint sau mỗi epoch.

        logging_strategy="steps",
        # logging_strategy:
        #   Ghi log theo số step.

        logging_steps=50,
        # logging_steps:
        #   Cứ 50 step ghi log một lần.

        save_total_limit=2,
        # save_total_limit:
        #   Chỉ giữ tối đa 2 checkpoint mới nhất để tiết kiệm dung lượng.

        warmup_ratio=0.05,
        # warmup_ratio:
        #   5% step đầu tăng learning rate từ nhỏ lên mức chính.
        #   Giúp train ổn định hơn.

        weight_decay=0.01,
        # weight_decay:
        #   Regularization nhẹ để giảm overfitting.

        report_to="none",
        # report_to:
        #   Không gửi log lên wandb/tensorboard.
    )

    if PREDICT_WITH_GENERATE_DURING_TRAINING:
        common_kwargs.update(
            load_best_model_at_end=True,
            # load_best_model_at_end:
            #   Cuối training tự load checkpoint tốt nhất.

            metric_for_best_model="rougeL",
            # metric_for_best_model:
            #   Dùng ROUGE-L để chọn checkpoint tốt nhất.

            greater_is_better=True,
            # greater_is_better:
            #   ROUGE càng cao càng tốt.
        )
    else:
        common_kwargs.update(
            load_best_model_at_end=False,
            # load_best_model_at_end:
            #   False vì khi không generate, Trainer không có rougeL để chọn best model.
        )

    # Transformers bản mới dùng eval_strategy, bản cũ dùng evaluation_strategy.
    sig = inspect.signature(Seq2SeqTrainingArguments.__init__)
    if "eval_strategy" in sig.parameters:
        common_kwargs["eval_strategy"] = "epoch"
    else:
        common_kwargs["evaluation_strategy"] = "epoch"

    return Seq2SeqTrainingArguments(**common_kwargs)


training_args = build_training_args()
training_args

Seq2SeqTrainingArguments(output_dir='/content/NLP/output/vit5-large-lora', overwrite_output_dir=False, do_train=True, do_eval=True, do_predict=False, evaluation_strategy=<IntervalStrategy.EPOCH: 'epoch'>, prediction_loss_only=False, per_device_train_batch_size=1, per_device_eval_batch_size=1, per_gpu_train_batch_size=None, per_gpu_eval_batch_size=None, gradient_accumulation_steps=8, eval_accumulation_steps=None, eval_delay=0, learning_rate=0.0002, weight_decay=0.01, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, max_grad_norm=1.0, num_train_epochs=5, max_steps=-1, lr_scheduler_type=<SchedulerType.LINEAR: 'linear'>, lr_scheduler_kwargs={}, warmup_ratio=0.05, warmup_steps=0, log_level='passive', log_level_replica='warning', log_on_each_node=True, logging_dir='/content/NLP/output/vit5-large-lora/runs/Jun06_06-26-35_e00bdafb244d', logging_strategy=<IntervalStrategy.STEPS: 'steps'>, logging_first_step=False, logging_steps=50, logging_nan_inf_filter=True, save_strategy=<IntervalStrate

## Cell 11 — Train và lưu LoRA adapter

**Nhiệm vụ:** Tạo `Seq2SeqTrainer`, fine-tune model, lưu adapter/tokenizer và kết quả train.

**Ý nghĩa:** Đây là cell huấn luyện chính. Nếu `DEBUG_MODE=True`, đây chỉ là chạy thử pipeline. Nếu `DEBUG_MODE=False`, đây là lần train thật.

In [11]:
# NHIỆM VỤ:
# - Tạo Trainer.
# - Train model.
# - Lưu LoRA adapter và tokenizer.

trainer = Seq2SeqTrainer(
    model=model,
    # model:
    #   ViT5-large đã gắn LoRA.

    args=training_args,
    # args:
    #   Toàn bộ cấu hình train/eval/save/log.

    train_dataset=tokenized_train,
    # train_dataset:
    #   Tập train đã tokenize.

    eval_dataset=tokenized_valid,
    # eval_dataset:
    #   Tập validation đã tokenize.

    tokenizer=tokenizer,
    # tokenizer:
    #   Dùng để decode/generate và lưu cùng adapter.

    data_collator=data_collator,
    # data_collator:
    #   Padding động theo batch.

    compute_metrics=compute_metrics if PREDICT_WITH_GENERATE_DURING_TRAINING else None,
    # compute_metrics:
    #   Chỉ dùng khi predict_with_generate=True.
)

# Chạy fine-tune.
train_result = trainer.train()

# Lưu LoRA adapter/checkpoint cuối.
trainer.save_model(OUTPUT_DIR)

# Lưu tokenizer để sau này load lại đúng tokenizer.
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved model/adapters to:", OUTPUT_DIR)
print("Files in output:", os.listdir(OUTPUT_DIR))

train_result

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
0,1.039500,0.935559
1,1.016100,0.890399
2,0.951300,0.870413
3,0.952500,0.863932
4,0.939600,0.858843


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a n

Saved model/adapters to: /content/NLP/output/vit5-large-lora
Files in output: ['tokenizer_config.json', 'adapter_model.safetensors', 'special_tokens_map.json', 'README.md', 'checkpoint-6730', 'training_args.bin', 'spiece.model', 'checkpoint-5387', 'added_tokens.json', 'adapter_config.json']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


TrainOutput(global_step=6730, training_loss=1.1566930592325786, metrics={'train_runtime': 28384.3335, 'train_samples_per_second': 1.898, 'train_steps_per_second': 0.237, 'total_flos': 1.3909748034403123e+17, 'train_loss': 1.1566930592325786, 'epoch': 5.0})

## Cell 12 — Hàm generate summary theo batch

**Nhiệm vụ:** Định nghĩa hàm sinh tóm tắt cho một danh sách article.

**Ý nghĩa:** Dùng chung cho validation, demo thủ công và sau này có thể dùng cho test set.

In [12]:
# NHIỆM VỤ:
# - Tạo hàm generate theo batch để inference nhanh hơn generate từng dòng.

def get_model_device(model):
    """Lấy device hiện tại của model.

    Parameters:
    - model:
        Model PyTorch hoặc PEFT model.

    Returns:
    - torch.device:
        Device của tham số đầu tiên trong model.
    """

    return next(model.parameters()).device


def generate_summaries(texts, batch_size=FINAL_GENERATE_BATCH_SIZE):
    """Sinh summary cho danh sách article.

    Parameters:
    - texts:
        List string chứa các article cần tóm tắt.
    - batch_size:
        Số article generate cùng lúc.
        Tăng batch_size giúp nhanh hơn nhưng tốn VRAM hơn.

    Returns:
    - List string chứa summary model sinh ra.
    """

    # Chuyển model sang eval để tắt dropout.
    model.eval()

    # Đưa model lên GPU nếu có.
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    predictions = []

    # Duyệt từng batch.
    for start_idx in range(0, len(texts), batch_size):
        batch_texts = texts[start_idx:start_idx + batch_size]

        # Thêm prefix nhiệm vụ và clean text.
        batch_inputs = [TASK_PREFIX + clean_text(text) for text in batch_texts]

        # Tokenize batch input.
        inputs = tokenizer(
            batch_inputs,
            max_length=MAX_SOURCE_LENGTH,
            # max_length:
            #   Giới hạn article đầu vào sau tokenize.

            truncation=True,
            # truncation:
            #   Cắt article nếu quá dài.

            padding=True,
            # padding:
            #   Padding các sample trong cùng batch về cùng độ dài.

            return_tensors="pt",
            # return_tensors:
            #   Trả tensor PyTorch.
        )

        # Đưa tensor lên cùng device với model.
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Inference không cần gradient.
        with torch.no_grad():
            output_ids = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],

                max_length=MAX_TARGET_LENGTH,
                # max_length:
                #   Độ dài tối đa của summary sinh ra.

                min_length=FINAL_MIN_LENGTH,
                # min_length:
                #   Độ dài tối thiểu để tránh summary quá ngắn.

                num_beams=FINAL_NUM_BEAMS,
                # num_beams:
                #   Số beam search. 4 thường cho chất lượng tốt hơn greedy.

                length_penalty=1.0,
                # length_penalty:
                #   1.0 là mặc định, không ưu tiên output quá ngắn/quá dài.

                no_repeat_ngram_size=FINAL_NO_REPEAT_NGRAM_SIZE,
                # no_repeat_ngram_size:
                #   Không cho lặp lại cụm n-gram, giảm repetition.

                early_stopping=True,
                # early_stopping:
                #   Dừng beam search khi các beam đã hoàn tất.
            )

        # Decode token IDs thành text.
        batch_predictions = tokenizer.batch_decode(
            output_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        predictions.extend([pred.strip() for pred in batch_predictions])

    return predictions

## Cell 13 — Sinh thử vài mẫu validation

**Nhiệm vụ:** In một vài article, summary chuẩn và summary model sinh.

**Ý nghĩa:** Đây là kiểm tra định tính nhanh sau training.

In [13]:
# NHIỆM VỤ:
# - Sinh thử tối đa 3 mẫu validation để nhìn chất lượng ban đầu.

NUM_QUICK_EXAMPLES = min(3, len(valid_df))

quick_articles = valid_df[SOURCE_COL].head(NUM_QUICK_EXAMPLES).tolist()
quick_references = valid_df[TARGET_COL].head(NUM_QUICK_EXAMPLES).tolist()
quick_predictions = generate_summaries(
    quick_articles,
    batch_size=min(FINAL_GENERATE_BATCH_SIZE, NUM_QUICK_EXAMPLES),
)

for i, (article, reference, prediction) in enumerate(
    zip(quick_articles, quick_references, quick_predictions),
    start=1,
):
    print("=" * 100)
    print(f"EXAMPLE {i}")
    print("\nARTICLE:")
    print(article[:800], "...")
    print("\nREFERENCE SUMMARY:")
    print(reference)
    print("\nMODEL PREDICTION:")
    print(prediction)

EXAMPLE 1

ARTICLE:
Giải thưởng công bố gần đây bởi World Travel Awards. Đây là năm thứ hai liên tiếp InterContinental Phu Quoc Long Beach Resort được vinh danh ở hạng mục gia đình trên toàn châu Á. Khu nghỉ dưỡng tọa lạc bên biển Phú Quốc, nổi bật với thiết kế lấy cảm hứng từ đại dương. Khuôn viên rộng rãi với nhiều mảng xanh thiên nhiên đậm chất nhiệt đới. Không gian sảnh lễ tân, phòng nghỉ, villa, nhà hàng... đều được chú trọng để tạo sự hài hòa với biển và cây cối. Du khách có thể chọn nghỉ ngơi tại khu phòng khách sạn rộng rãi, tiện nghi, hoặc những căn hộ, phòng suite, biệt thự cao cấp hướng biển. Mỗi không gian được thiết kế dựa trên tinh thần gắn kết các thành viên trong gia đình. Bên cạnh tiện nghi sang trọng, InterContinental Phu Quoc còn hút khách gia đình nhờ loạt trải nghiệm giải trí, thư giãn đa ...

REFERENCE SUMMARY:
InterContinental Phu Quoc Long Beach Resort đã được vinh danh là khu nghỉ dưỡng gia đình tốt nhất châu Á lần thứ 2 liên tiếp. Khu nghỉ dưỡng này tọa lạc bê

## Cell 14 — Generate prediction trên validation set

**Nhiệm vụ:** Sinh prediction cho validation set hoặc một phần validation set rồi lưu CSV.

**Ý nghĩa:** File này dùng để tính ROUGE/BLEU và phân tích lỗi trong báo cáo.

In [14]:
# NHIỆM VỤ:
# - Generate prediction trên validation.
# - Lưu article, reference, prediction ra CSV.

from tqdm.auto import tqdm  # Thanh tiến trình.

# Quyết định số mẫu validation dùng để generate.
# Nếu VALIDATION_MAX_GENERATE_SAMPLES=None:
#   - DEBUG_MODE=True: lấy tối đa 50 mẫu để chạy nhanh.
#   - DEBUG_MODE=False: generate toàn bộ validation.
if VALIDATION_MAX_GENERATE_SAMPLES is None:
    max_generate_samples = min(50, len(valid_df)) if DEBUG_MODE else len(valid_df)
else:
    max_generate_samples = min(VALIDATION_MAX_GENERATE_SAMPLES, len(valid_df))

# Lấy validation subset.
gen_df = valid_df.iloc[:max_generate_samples].copy().reset_index(drop=True)

print("Validation samples for generation:", len(gen_df))

# Lấy article/reference.
validation_articles = gen_df[SOURCE_COL].tolist()
validation_references = gen_df[TARGET_COL].tolist()

# Generate theo batch.
validation_predictions = []
for start_idx in tqdm(range(0, len(validation_articles), FINAL_GENERATE_BATCH_SIZE)):
    batch_articles = validation_articles[start_idx:start_idx + FINAL_GENERATE_BATCH_SIZE]
    batch_predictions = generate_summaries(
        batch_articles,
        batch_size=FINAL_GENERATE_BATCH_SIZE,
    )
    validation_predictions.extend(batch_predictions)

# Tạo DataFrame kết quả.
pred_df = pd.DataFrame({
    "article": validation_articles,
    "reference": validation_references,
    "prediction": validation_predictions,
})

# Lưu CSV.
# encoding="utf-8-sig": mở bằng Excel ít lỗi tiếng Việt hơn.
pred_df.to_csv(PREDICTION_CSV, index=False, encoding="utf-8-sig")

print("Saved validation predictions to:", PREDICTION_CSV)
display(pred_df.head())

Validation samples for generation: 1349


  0%|          | 0/338 [00:00<?, ?it/s]

Saved validation predictions to: /content/NLP/output/vit5_large_validation_predictions.csv


,article,reference,prediction
0,Giải thưởng công bố gần đây bởi World Travel A...,InterContinental Phu Quoc Long Beach Resort đã...,"Công bố gần đây bởi World Travel Awards, Inter..."
1,Theo bảng xếp hạng 20 quốc gia tốt nhất thế gi...,Việt Nam đã xếp hạng 15 trên bảng xếp hạng 20 ...,Bộ xếp hạng 20 quốc gia tốt nhất thế giới năm ...
2,"Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Bài viết giới thiệu về Ngày hội Văn hóa, Thể t..."
3,Giải thưởng do Tạp chí du lịch Condé Nast Trav...,"Phú Quốc, đảo ngọc của Việt Nam, đã được vinh ...",Bộ du lịch Việt Nam đã vinh danh Phú Quốc vào ...
4,KKday Vietnam vừa công bố hợp tác chiến lược S...,KKday Vietnam vừa công bố hợp tác chiến lược v...,Công ty KKday Vietnam vừa công bố hợp tác chiế...


## Cell 15 — Tính ROUGE và BLEU trên validation

**Nhiệm vụ:** Tính metric chính trên prediction đã generate.

**Ý nghĩa:** ROUGE thường dùng cho tóm tắt; BLEU chỉ nên xem là metric tham khảo.

In [15]:
# NHIỆM VỤ:
# - Tính ROUGE và BLEU trên validation prediction.
# - Lưu metric ra JSON.

# Chuẩn bị list prediction/reference.
predictions = pred_df["prediction"].fillna("").astype(str).tolist()
references = pred_df["reference"].fillna("").astype(str).tolist()

# Tính ROUGE.
rouge_result = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=False,
    # use_stemmer:
    #   False vì stemming của rouge-score chủ yếu thiết kế cho tiếng Anh.
)

# Tính BLEU bằng SacreBLEU.
bleu_metric = evaluate.load("sacrebleu")
bleu_result = bleu_metric.compute(
    predictions=predictions,
    references=[[ref] for ref in references],
    # references:
    #   SacreBLEU yêu cầu dạng list of list.
    #   Mỗi prediction có thể có nhiều reference; ở đây mỗi mẫu có 1 reference.
)

# Gom metric.
validation_metrics = {
    "num_samples": len(pred_df),
    "debug_mode": DEBUG_MODE,
    "rouge": rouge_result,
    "bleu": bleu_result["score"],
}

# In metric dễ đọc.
print("===== VALIDATION ROUGE =====")
for key, value in rouge_result.items():
    print(f"{key}: {value:.4f}")

print("\n===== VALIDATION BLEU =====")
print(f"BLEU: {bleu_result['score']:.4f}")

# Lưu JSON.
with open(METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(
        validation_metrics,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("\nSaved validation metrics to:", METRICS_JSON)

===== VALIDATION ROUGE =====
rouge1: 0.7343
rouge2: 0.4625
rougeL: 0.4812
rougeLsum: 0.4810

===== VALIDATION BLEU =====
BLEU: 32.0709

Saved validation metrics to: /content/NLP/output/vit5_large_validation_metrics.json


## Cell 16 — Tính BERTScore tùy chọn

**Nhiệm vụ:** Tính BERTScore nếu cần đánh giá gần nghĩa hơn ROUGE/BLEU.

**Ý nghĩa:** Cell này mặc định tắt vì chạy chậm và tải thêm model embedding.

In [16]:
# NHIỆM VỤ:
# - Tính BERTScore nếu RUN_BERTSCORE=True.

if RUN_BERTSCORE:
    from bert_score import score

    # Lấy prediction/reference.
    preds = pred_df["prediction"].fillna("").astype(str).tolist()
    refs = pred_df["reference"].fillna("").astype(str).tolist()

    # Tính BERTScore.
    P, R, F1 = score(
        preds,
        refs,

        model_type="bert-base-multilingual-cased",
        # model_type:
        #   Model multilingual để tính embedding cho tiếng Việt.
        #   Có thể thay bằng model tiếng Việt nếu muốn tối ưu hơn.

        verbose=True,
        # verbose:
        #   In tiến trình khi chạy.
    )

    # Tính trung bình.
    bertscore_result = {
        "bertscore_precision": float(P.mean().item()),
        "bertscore_recall": float(R.mean().item()),
        "bertscore_f1": float(F1.mean().item()),
    }

    print("===== VALIDATION BERTSCORE =====")
    print(bertscore_result)

    # Lưu BERTScore.
    with open(BERTSCORE_JSON, "w", encoding="utf-8") as f:
        json.dump(
            bertscore_result,
            f,
            ensure_ascii=False,
            indent=2,
        )

    print("Saved BERTScore to:", BERTSCORE_JSON)
else:
    print("RUN_BERTSCORE=False, skipped.")

RUN_BERTSCORE=False, skipped.


## Cell 17 — Lưu ví dụ định tính cho báo cáo

**Nhiệm vụ:** Lấy một số mẫu validation gồm article, summary chuẩn và prediction của model.

**Ý nghĩa:** Dùng cho phần qualitative evaluation/error analysis trong báo cáo.

In [17]:
# NHIỆM VỤ:
# - Lấy ví dụ định tính từ pred_df.
# - Lưu ra CSV để đưa vào báo cáo.

# Số ví dụ muốn lấy.
# min(10, len(pred_df)): tránh lỗi nếu pred_df ít hơn 10 dòng.
NUM_EXAMPLES_FOR_REPORT = min(10, len(pred_df))

# sample(..., random_state=42): lấy mẫu ngẫu nhiên nhưng tái lập được.
examples_df = pred_df.sample(
    n=NUM_EXAMPLES_FOR_REPORT,
    random_state=42,
).copy()

# Lưu CSV.
examples_df.to_csv(EXAMPLES_CSV, index=False, encoding="utf-8-sig")

print("Saved examples to:", EXAMPLES_CSV)
display(examples_df)

Saved examples to: /content/NLP/output/vit5_large_examples_for_report.csv


,article,reference,prediction
289,"TheoThe Paper, khán giả vốn quen Giả Linh với ...","Diễn viên Giả Linh, nổi tiếng với hình tượng m...","Một năm qua, Giả Linh đã giảm cân thành công t..."
1036,"Phút 43, trận đấu trên sân Sultan Ibrahim (ban...",Trong trận đấu giữa Buriram United (Thái Lan) ...,Bài viết mô tả trận đấu giữa Buriram United và...
535,Ngân hàng Trung ương Nga cuối tuần trước thông...,Ngân hàng Trung ương Nga vừa thông báo giá trị...,Bộ Tài chính Nga vừa thông báo giá trị khối và...
346,"Theo đại diện tuyển sinh các trường, đây là th...",Ngày 17/8 là thời điểm kết thúc việc xử lý ngu...,Bộ Giáo dục và Đào tạo yêu cầu các trường đại ...
1075,"""Có nhiều thứ của Liverpool khiến tôi nhớ đến ...",Gary Neville ca ngợi sự đa dạng chiến thuật củ...,Một cựu hậu vệ người Anh cho rằng Liverpool đa...
782,Ngoài chính sách giảm 50% lệ phí trước bạ từ C...,Chính phủ đang áp dụng chính sách giảm 50% lệ ...,"Dự kiến, người dùng mua các dòng xe BMW lắp rá..."
303,Con trai họa sĩ - ông Hồ Hồng Lĩnh - cho biết ...,Họa sĩ Hồ Hữu Thủ sinh năm 1940 tại Thủ Dầu Mộ...,"Văn bản trên cho biết họa sĩ Hồ Hữu Thủ, một d..."
754,Mẫu concept EV Fun chưa phải là phiên bản sản ...,Honda đã trình diễn hai mẫu xe điện tại triển ...,Một mẫu concept EV Fun chưa phải là phiên bản ...
536,"Chốt phiên giao dịch 11/11, giá vàng thế giới ...","Sau khi ông Donald Trump đắc cử Tổng thống Mỹ,...","Một phiên giao dịch 11/11, giá vàng thế giới g..."
76,Khoảng 200 con khỉ hôm 16/11 lang thang trên đ...,Khoảng 200 con khỉ lang thang trên đường phố v...,Bộ trưởng Somchai Seedee cho biết khoảng 200 c...


## Các cell test set độc lập — Đã comment toàn bộ vì chưa có `test.parquet`

**Vị trí đặt:** Sau khi model đã train xong và đã có hàm `generate_summaries`.

**Cách dùng sau này:** Khi bạn có `/content/NLP/data/test.parquet`, bỏ dấu `#` ở các dòng trong các cell bên dưới rồi chạy từ trên xuống.

**Lưu ý quan trọng:** Validation dùng trong quá trình train/điều chỉnh; test set chỉ dùng một lần cuối để đánh giá khách quan.

### Cell Test 1 — Cấu hình và đọc `test.parquet`

In [18]:
# # ============================================================
# # OPTIONAL TEST CELL 1: Cấu hình và đọc test.parquet
# # ============================================================
# # NHIỆM VỤ:
# # - Khai báo đường dẫn test set độc lập.
# # - Đọc test.parquet.
# # - Làm sạch giống train/validation.
# #
# # Vì hiện tại chưa có test.parquet nên toàn bộ cell này được comment.
# # Khi có file test, bỏ dấu # ở các dòng cần chạy.
#
# TEST_PATH = f"{DATA_DIR}/test.parquet"
# # TEST_PATH:
# #   Đường dẫn file test độc lập.
# #   File này nên có cùng schema với train/val: article và summary.
#
# TEST_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/test_evaluation"
# # TEST_OUTPUT_DIR:
# #   Thư mục lưu prediction/metric trên test set.
#
# TEST_PREDICTION_CSV = f"{TEST_OUTPUT_DIR}/test_predictions.csv"
# # TEST_PREDICTION_CSV:
# #   File CSV lưu article, reference, prediction trên test.
#
# TEST_METRICS_JSON = f"{TEST_OUTPUT_DIR}/test_metrics.json"
# # TEST_METRICS_JSON:
# #   File JSON lưu ROUGE/BLEU/BERTScore trên test.
#
# TEST_MAX_SAMPLES = None
# # TEST_MAX_SAMPLES:
# #   None nghĩa là đánh giá toàn bộ test set.
# #   Có thể đặt 100 hoặc 200 nếu muốn chạy thử nhanh.
#
# TEST_BATCH_SIZE = 4
# # TEST_BATCH_SIZE:
# #   Batch size khi generate trên test.
# #   Nếu T4 bị OOM, giảm xuống 2 hoặc 1.
#
# os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)
#
# if not os.path.exists(TEST_PATH):
#     raise FileNotFoundError(f"Chưa tìm thấy test file: {TEST_PATH}")
#
# test_df = load_and_clean_parquet(TEST_PATH)
#
# if TEST_MAX_SAMPLES is not None:
#     test_df = test_df.head(TEST_MAX_SAMPLES).reset_index(drop=True)
#
# print("Test used:", test_df.shape)
# display(test_df.head())

### Cell Test 2 — Kiểm tra trùng lặp train/validation/test

In [19]:
# # ============================================================
# # OPTIONAL TEST CELL 2: Kiểm tra data leakage
# # ============================================================
# # NHIỆM VỤ:
# # - Kiểm tra article trong test có trùng train hoặc validation không.
# # - Nếu trùng nhiều, metric test sẽ không còn khách quan.
#
# def normalize_for_leakage_check(text):
#     """Chuẩn hóa nhẹ text để so sánh trùng lặp."""
#     return " ".join(str(text).lower().strip().split())
#
# train_articles = set(train_df[SOURCE_COL].map(normalize_for_leakage_check))
# # train_articles:
# #   Tập article train đã normalize để kiểm tra trùng.
#
# valid_articles = set(valid_df[SOURCE_COL].map(normalize_for_leakage_check))
# # valid_articles:
# #   Tập article validation đã normalize.
#
# test_articles = set(test_df[SOURCE_COL].map(normalize_for_leakage_check))
# # test_articles:
# #   Tập article test đã normalize.
#
# overlap_train_test = len(train_articles.intersection(test_articles))
# # overlap_train_test:
# #   Số article test bị trùng train.
#
# overlap_valid_test = len(valid_articles.intersection(test_articles))
# # overlap_valid_test:
# #   Số article test bị trùng validation.
#
# print("Số article test trùng train:", overlap_train_test)
# print("Số article test trùng validation:", overlap_valid_test)
# print("Tỷ lệ trùng train-test:", overlap_train_test / max(len(test_articles), 1))
# print("Tỷ lệ trùng valid-test:", overlap_valid_test / max(len(test_articles), 1))

### Cell Test 3 — Generate prediction trên test set

In [20]:
# # ============================================================
# # OPTIONAL TEST CELL 3: Generate prediction trên test set
# # ============================================================
# # NHIỆM VỤ:
# # - Dùng model fine-tuned để sinh summary cho test set.
# # - Không cập nhật trọng số model.
#
# test_articles = test_df[SOURCE_COL].tolist()
# # test_articles:
# #   Danh sách article test đầu vào.
#
# test_references = test_df[TARGET_COL].tolist()
# # test_references:
# #   Danh sách summary chuẩn của test.
#
# test_predictions = []
# # test_predictions:
# #   Nơi lưu summary do model sinh ra.
#
# for start_idx in tqdm(range(0, len(test_articles), TEST_BATCH_SIZE)):
#     batch_articles = test_articles[start_idx:start_idx + TEST_BATCH_SIZE]
#     # batch_articles:
#     #   Một batch article để generate.
#
#     batch_predictions = generate_summaries(
#         batch_articles,
#         batch_size=TEST_BATCH_SIZE,
#     )
#     # batch_predictions:
#     #   Summary sinh ra cho batch hiện tại.
#
#     test_predictions.extend(batch_predictions)
#
# test_pred_df = pd.DataFrame({
#     "article": test_articles,
#     "reference": test_references,
#     "prediction": test_predictions,
# })
#
# test_pred_df.to_csv(TEST_PREDICTION_CSV, index=False, encoding="utf-8-sig")
#
# print("Saved test predictions to:", TEST_PREDICTION_CSV)
# display(test_pred_df.head())

### Cell Test 4 — Tính ROUGE/BLEU/BERTScore trên test set

In [21]:
# # ============================================================
# # OPTIONAL TEST CELL 4: Tính metric test
# # ============================================================
# # NHIỆM VỤ:
# # - Tính ROUGE và BLEU trên test set.
# # - Tùy chọn tính BERTScore.
#
# test_predictions_list = test_pred_df["prediction"].fillna("").astype(str).tolist()
# # test_predictions_list:
# #   Danh sách summary model sinh ra.
#
# test_references_list = test_pred_df["reference"].fillna("").astype(str).tolist()
# # test_references_list:
# #   Danh sách summary chuẩn.
#
# test_rouge = rouge.compute(
#     predictions=test_predictions_list,
#     references=test_references_list,
#     use_stemmer=False,
# )
# # test_rouge:
# #   ROUGE-1, ROUGE-2, ROUGE-L, ROUGE-Lsum trên test.
#
# test_bleu = bleu_metric.compute(
#     predictions=test_predictions_list,
#     references=[[ref] for ref in test_references_list],
# )
# # test_bleu:
# #   BLEU trên test, chỉ dùng tham khảo.
#
# RUN_TEST_BERTSCORE = False
# # RUN_TEST_BERTSCORE:
# #   True để tính BERTScore test.
# #   False để bỏ qua vì chạy chậm.
#
# test_metrics = {
#     "num_samples": len(test_pred_df),
#     "rouge": test_rouge,
#     "bleu": test_bleu["score"],
# }
#
# if RUN_TEST_BERTSCORE:
#     from bert_score import score
#
#     P, R, F1 = score(
#         test_predictions_list,
#         test_references_list,
#         model_type="bert-base-multilingual-cased",
#         verbose=True,
#     )
#
#     test_metrics["bertscore"] = {
#         "precision": float(P.mean().item()),
#         "recall": float(R.mean().item()),
#         "f1": float(F1.mean().item()),
#     }
#
# print("===== TEST METRICS =====")
# print(json.dumps(test_metrics, ensure_ascii=False, indent=2))
#
# with open(TEST_METRICS_JSON, "w", encoding="utf-8") as f:
#     json.dump(test_metrics, f, ensure_ascii=False, indent=2)
#
# print("Saved test metrics to:", TEST_METRICS_JSON)

### Cell Test 5 — Xem ví dụ định tính trên test set

In [22]:
# # ============================================================
# # OPTIONAL TEST CELL 5: Xem ví dụ prediction trên test set
# # ============================================================
# # NHIỆM VỤ:
# # - Xem trực tiếp model tóm tắt như thế nào trên test.
# # - Giúp phát hiện lỗi copy nguyên văn, sai ý, thiếu ý chính hoặc lặp từ.
#
# NUM_TEST_EXAMPLES_TO_SHOW = 5
# # NUM_TEST_EXAMPLES_TO_SHOW:
# #   Số ví dụ test muốn in ra màn hình.
#
# for i in range(min(NUM_TEST_EXAMPLES_TO_SHOW, len(test_pred_df))):
#     print("=" * 100)
#     print(f"TEST EXAMPLE {i + 1}")
#
#     print("\nARTICLE:")
#     print(test_pred_df.loc[i, "article"][:1500])
#
#     print("\nREFERENCE SUMMARY:")
#     print(test_pred_df.loc[i, "reference"])
#
#     print("\nMODEL PREDICTION:")
#     print(test_pred_df.loc[i, "prediction"])

## Cell 18 — Demo với văn bản tự nhập

**Nhiệm vụ:** Nhập một article bất kỳ và xem model tóm tắt.

**Ý nghĩa:** Cell này dùng để demo nhanh, không ảnh hưởng metric chính.

In [25]:
# NHIỆM VỤ:
# - Chạy thử model với một văn bản tự nhập.

# @title Chạy thử mô hình với văn bản của bạn

custom_article = "Redmi Note 12 Turbo là một trong những mẫu smartphone tầm trung nổi bật nhất của Xiaomi khi ra mắt vào tháng 3 năm 2023. Đây cũng là chiếc điện thoại đầu tiên trên thế giới được trang bị vi xử lý Snapdragon 7+ Gen 2, con chip mang lại hiệu năng mạnh mẽ và tiệm cận với nhiều thiết bị cao cấp trên thị trường. Nhờ được sản xuất trên tiến trình 4nm của TSMC, bộ xử lý này không chỉ cho khả năng vận hành mượt mà mà còn tối ưu điện năng hiệu quả. Người dùng có thể dễ dàng thực hiện các tác vụ hằng ngày như lướt web, xem phim, làm việc trực tuyến hay chơi những tựa game đồ họa nặng như PUBG Mobile, Call of Duty Mobile hoặc Genshin Impact với mức thiết lập cao mà vẫn duy trì được độ ổn định tốt. Bên cạnh hiệu năng mạnh mẽ, Redmi Note 12 Turbo còn được trang bị màn hình OLED kích thước 6,67 inch với độ phân giải Full HD+, tần số quét 120Hz cùng công nghệ HDR10+ và Dolby Vision. Nhờ đó, thiết bị mang đến chất lượng hiển thị sắc nét, màu sắc sống động và trải nghiệm vuốt chạm mượt mà trong mọi tác vụ. Độ sáng cao của màn hình cũng giúp người dùng dễ dàng sử dụng điện thoại trong điều kiện ánh sáng mạnh ngoài trời. Về khả năng chụp ảnh, Redmi Note 12 Turbo sở hữu cụm camera sau gồm cảm biến chính 64MP hỗ trợ chống rung quang học OIS, camera góc siêu rộng 8MP và camera macro 2MP. Hệ thống camera này cho phép chụp được những bức ảnh có độ chi tiết tốt, khả năng xử lý trong điều kiện thiếu sáng khá ổn và hỗ trợ quay video 4K. Ngoài ra, tính năng zoom 2x chất lượng cao cũng giúp người dùng linh hoạt hơn trong việc ghi lại các khoảnh khắc hằng ngày. Máy còn được trang bị viên pin dung lượng 5.000mAh kết hợp công nghệ sạc nhanh 67W, giúp đáp ứng tốt nhu cầu sử dụng liên tục trong cả ngày dài và rút ngắn đáng kể thời gian chờ sạc. Một điểm mạnh khác của Redmi Note 12 Turbo là bộ nhớ lớn với các tùy chọn lên đến 16GB RAM LPDDR5 và 1TB bộ nhớ trong UFS 3.1, mang lại khả năng đa nhiệm mượt mà và không gian lưu trữ rộng rãi cho hình ảnh, video, tài liệu hay các ứng dụng dung lượng lớn. Ngoài ra, hệ thống loa kép stereo hỗ trợ Dolby Atmos cùng jack cắm tai nghe 3.5mm cũng góp phần nâng cao trải nghiệm giải trí của người dùng. Tuy nhiên, dù sở hữu nhiều ưu điểm nổi bật, Redmi Note 12 Turbo không còn là lựa chọn thực sự hấp dẫn trong năm 2025. Nguyên nhân chính là thiết bị không được Xiaomi phân phối chính hãng tại Việt Nam mà chủ yếu xuất hiện dưới dạng hàng xách tay hoặc hàng nhập khẩu. Điều này khiến người dùng gặp nhiều hạn chế liên quan đến chế độ bảo hành, hỗ trợ kỹ thuật và cập nhật phần mềm. Trong trường hợp thiết bị gặp lỗi phần cứng hoặc cần sửa chữa, việc tìm kiếm linh kiện thay thế và nhận hỗ trợ cũng có thể gặp khó khăn hơn so với các sản phẩm chính hãng. Mặc dù mức giá của Redmi Note 12 Turbo hiện nay khá cạnh tranh, nhưng khi cân nhắc đến các yếu tố sử dụng lâu dài, đây không còn là sự lựa chọn tối ưu cho đa số người dùng. Thay vào đó, các mẫu Redmi Note thế hệ mới được phân phối chính hãng tại Việt Nam sẽ mang lại sự an tâm hơn nhờ được hỗ trợ phần mềm mới, chế độ bảo hành đầy đủ và nhiều cải tiến về công nghệ. Có thể nói rằng Redmi Note 12 Turbo từng là một trong những smartphone tầm trung đáng mua nhất khi ra mắt nhờ hiệu năng mạnh, màn hình đẹp, pin lớn và khả năng sạc nhanh." # @param {type:"string"}
# custom_article:
#   Văn bản đầu vào tự nhập.
# @param {type:"string"}:
#   Cú pháp Colab để hiện ô nhập string trên giao diện.

print("Đang tạo tóm tắt...")

custom_prediction = generate_summaries(
    [custom_article],
    batch_size=1,
)[0]

print("\n" + "=" * 80)
print("📝 BÀI VIẾT GỐC:")
print(custom_article)
print("-" * 80)
print("✨ TÓM TẮT CỦA MÔ HÌNH:")
print(custom_prediction)
print("=" * 80)

Đang tạo tóm tắt...

📝 BÀI VIẾT GỐC:
Redmi Note 12 Turbo là một trong những mẫu smartphone tầm trung nổi bật nhất của Xiaomi khi ra mắt vào tháng 3 năm 2023. Đây cũng là chiếc điện thoại đầu tiên trên thế giới được trang bị vi xử lý Snapdragon 7+ Gen 2, con chip mang lại hiệu năng mạnh mẽ và tiệm cận với nhiều thiết bị cao cấp trên thị trường. Nhờ được sản xuất trên tiến trình 4nm của TSMC, bộ xử lý này không chỉ cho khả năng vận hành mượt mà mà còn tối ưu điện năng hiệu quả. Người dùng có thể dễ dàng thực hiện các tác vụ hằng ngày như lướt web, xem phim, làm việc trực tuyến hay chơi những tựa game đồ họa nặng như PUBG Mobile, Call of Duty Mobile hoặc Genshin Impact với mức thiết lập cao mà vẫn duy trì được độ ổn định tốt. Bên cạnh hiệu năng mạnh mẽ, Redmi Note 12 Turbo còn được trang bị màn hình OLED kích thước 6,67 inch với độ phân giải Full HD+, tần số quét 120Hz cùng công nghệ HDR10+ và Dolby Vision. Nhờ đó, thiết bị mang đến chất lượng hiển thị sắc nét, màu sắc sống động và trải n

## Cell 19 — Nén output để tải về

**Nhiệm vụ:** Gom adapter, tokenizer, prediction CSV, metric JSON và ví dụ báo cáo vào một file `.zip`.

**Ý nghĩa:** Sau khi chạy xong trên Colab, tải file zip này để lưu local, nộp bài hoặc đưa lên GitHub.

In [24]:
# NHIỆM VỤ:
# - Nén các file/thư mục output quan trọng.

import zipfile

# Chuyển đường dẫn sang Path object.
output_base = Path(OUTPUT_BASE_DIR)
zip_path = Path(ZIP_PATH)

# Danh sách item cần zip.
items_to_zip = [
    Path(OUTPUT_DIR),
    # OUTPUT_DIR:
    #   Thư mục chứa LoRA adapter và tokenizer.

    Path(PREDICTION_CSV),
    # PREDICTION_CSV:
    #   Prediction validation.

    Path(METRICS_JSON),
    # METRICS_JSON:
    #   ROUGE/BLEU validation.

    Path(BERTSCORE_JSON),
    # BERTSCORE_JSON:
    #   BERTScore validation nếu có.

    Path(EXAMPLES_CSV),
    # EXAMPLES_CSV:
    #   Ví dụ định tính cho báo cáo.

    Path(f"{OUTPUT_BASE_DIR}/test_evaluation"),
    # test_evaluation:
    #   Thư mục test nếu sau này bạn bỏ comment các cell test.
]

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    # compression:
    #   Nén file để giảm dung lượng tải về.
) as zf:
    for item in items_to_zip:
        # Bỏ qua item chưa tồn tại.
        if not item.exists():
            continue

        if item.is_dir():
            # Nếu là thư mục, zip toàn bộ file bên trong.
            for file_path in item.rglob("*"):
                if file_path.is_file() and file_path != zip_path:
                    zf.write(
                        file_path,
                        arcname=file_path.relative_to(output_base),
                        # arcname:
                        #   Đường dẫn tương đối bên trong file zip.
                    )
        else:
            # Nếu là file đơn lẻ, zip trực tiếp.
            zf.write(
                item,
                arcname=item.relative_to(output_base),
            )

print("Output zip:", zip_path)

Output zip: /content/NLP/output/vit5_large_outputs.zip
